In [1]:
%load_ext autoreload
%autoreload 2

Failed to read module file 'c:\Users\viniciuscosta\AppData\Local\Programs\Python\Python313\Lib\shlex.py' for module 'shlex': UnicodeDecodeError
Traceback (most recent call last):
  File "C:\Users\viniciuscosta\AppData\Roaming\Python\Python313\site-packages\IPython\core\extensions.py", line 62, in load_extension
    return self._load_extension(module_str)
           ~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^
  File "C:\Users\viniciuscosta\AppData\Roaming\Python\Python313\site-packages\IPython\core\extensions.py", line 77, in _load_extension
    mod = import_module(module_str)
  File "c:\Users\viniciuscosta\AppData\Local\Programs\Python\Python313\Lib\importlib\__init__.py", line 88, in import_module
    return _bootstrap._gcd_import(name[level:], package, level)
           ~~~~~~~~~~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 1387, in _gcd_import
  File "<frozen importlib._bootstrap>", line 1360, in _find_and_load
  File "<frozen importlib._bootstrap>", 

In [2]:
import sys
sys.path.append('../')

In [3]:
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
from tqdm import tqdm
import os
import numpy as np
import random 
import torch

from src.data_extraction.labels import get_label_metadata
from src.data_extraction.attributions import get_attributions_metadata
from src.data_extraction.video_frame import (create_video_frame_metadata_from_label_and_attributions,
                                             create_video_frame_df, 
                                             save_video_frame_metadata_to_csv)
from src.data_extraction.patients import (load_patients_metadata_from_csv,
                                          get_patients_metadata_from_reindex_file,
                                          save_patients_metadata_to_csv)
from src.create_folders import get_frame_from_video

In [4]:
def set_seed(seed_value=42):
    random.seed(seed_value)
    os.environ['PYTHONHASHSEED'] = str(seed_value)
    np.random.seed(seed_value)
    torch.manual_seed(seed_value)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed_value)

set_seed(42)

In [5]:
# Locais de salvamento/acesso dos dados

videos_dir = '..\\data\\videos\\' 
labels_dir = '..\\data\\rotulos\\anotacoes-tecgraf\\'
frames_dir = '..\\data\\frames\\'

video_reindex_csv_path = '..\\data\\metadados\\video_dictionary.csv'
patients_metadata_path = '..\\data\\metadados\\patients_metadata.csv'

### **Coletando e Filtrando dados**

In [6]:
# Coletando Informações dos Rótulos feitos

attribution_metadata_df = get_attributions_metadata(labels_dir)
labels_metadata_df = get_label_metadata(labels_dir)

video_frame_metadata_df = create_video_frame_metadata_from_label_and_attributions(
    label_metadata=labels_metadata_df,
    attributions_metadata=attribution_metadata_df
)
video_frame_metadata_df

The results CSV is missing the following required columns: {'Height', 'Width', 'BX', 'BY'}. Check the file at ..\data\rotulos\anotacoes-tecgraf\batch-2\VR\v98_f45\Results.csv.
The results CSV is missing the following required columns: {'Angle'}. Check the file at ..\data\rotulos\anotacoes-tecgraf\batch-3\CS\v101_f74\Results.csv.
Shape of attributions metadata df: (2000, 5)
Shape of label metadata df: (1985, 7)
Shape of labeled frames df: (2000, 9)


,batch,labeler,video_frame,video_id,frame_id,file_path,has_mask,has_points
0,1,AM,v33_f24,33,24,..\data\rotulos\anotacoes-tecgraf\batch-1\AM\v...,True,True
1,1,AM,v54_f127,54,127,..\data\rotulos\anotacoes-tecgraf\batch-1\AM\v...,True,True
2,1,AM,v74_f65,74,65,..\data\rotulos\anotacoes-tecgraf\batch-1\AM\v...,True,True
3,1,BB,v16_f122,16,122,..\data\rotulos\anotacoes-tecgraf\batch-1\BB\v...,True,True
4,1,BB,v66_f166,66,166,..\data\rotulos\anotacoes-tecgraf\batch-1\BB\v...,True,True
...,...,...,...,...,...,...,...,...
1995,5,VR,v70_f137,70,137,..\data\rotulos\anotacoes-tecgraf\batch-5\VR\v...,True,True
1996,5,VR,v76_f80,76,80,..\data\rotulos\anotacoes-tecgraf\batch-5\VR\v...,True,True
1997,5,VR,v78_f213,78,213,..\data\rotulos\anotacoes-tecgraf\batch-5\VR\v...,True,True
1998,5,VR,v89_f118,89,118,..\data\rotulos\anotacoes-tecgraf\batch-5\VR\v...,True,True


In [7]:
# Coleta Informações dos Pacientes

if os.path.exists(patients_metadata_path):
    print(f"Patients metadata file already exists at: {patients_metadata_path}. No need to regenerate.")

else:
    if not os.path.exists(video_reindex_csv_path):
        raise FileNotFoundError(f"Video reindex file not found: {video_reindex_csv_path}. Please provide the correct path to generate patients metadata.")

    patients_metadata_df = get_patients_metadata_from_reindex_file(video_reindex_csv_path)
    save_patients_metadata_to_csv(patients_metadata_df)

Patients metadata file already exists at: ..\data\metadados\patients_metadata.csv. No need to regenerate.


In [8]:
# Carrega dados dos Pacientes
patients_df = load_patients_metadata_from_csv()

In [9]:
# Selecionando somente os dados de interesse

video_frame_df = create_video_frame_df(
    videos_dir=videos_dir,
    labels_dir=labels_dir,
    frames_dir=frames_dir,
    batch=None, # Get all batches
    labeler=None, # Get all labelers
    labeler_filter_criteria="union", 
    target='points'
)

video_frame_df

The results CSV is missing the following required columns: {'Height', 'Width', 'BX', 'BY'}. Check the file at ..\data\rotulos\anotacoes-tecgraf\batch-2\VR\v98_f45\Results.csv.
The results CSV is missing the following required columns: {'Angle'}. Check the file at ..\data\rotulos\anotacoes-tecgraf\batch-3\CS\v101_f74\Results.csv.
Shape of attributions metadata df: (2000, 5)
Shape of label metadata df: (1985, 7)
Shape of labeled frames df: (2000, 9)


,video_frame,video_id,frame_id,batch,fonte_dados,paciente_id,momento,procedimento,selected_labeler,video_path,frame_path,target_dir
0,v100_f10,100,10,5,video100,115,pos,total,CS,..\data\videos\100.avi,..\data\frames\v100_f10.png,..\data\rotulos\anotacoes-tecgraf\batch-5\CS\v...
1,v100_f12,100,12,2,video100,115,pos,total,BB,..\data\videos\100.avi,..\data\frames\v100_f12.png,..\data\rotulos\anotacoes-tecgraf\batch-2\BB\v...
2,v100_f13,100,13,3,video100,115,pos,total,AM,..\data\videos\100.avi,..\data\frames\v100_f13.png,..\data\rotulos\anotacoes-tecgraf\batch-3\AM\v...
3,v100_f14,100,14,4,video100,115,pos,total,AM,..\data\videos\100.avi,..\data\frames\v100_f14.png,..\data\rotulos\anotacoes-tecgraf\batch-4\AM\v...
4,v100_f15,100,15,5,video100,115,pos,total,AM,..\data\videos\100.avi,..\data\frames\v100_f15.png,..\data\rotulos\anotacoes-tecgraf\batch-5\AM\v...
...,...,...,...,...,...,...,...,...,...,...,...,...
968,v99_f7,99,7,2,video100,114,pos,total,AM,..\data\videos\99.avi,..\data\frames\v99_f7.png,..\data\rotulos\anotacoes-tecgraf\batch-2\AM\v...
969,v9_f118,9,118,5,video100,24,pos,total,CS,..\data\videos\9.avi,..\data\frames\v9_f118.png,..\data\rotulos\anotacoes-tecgraf\batch-5\CS\v...
970,v9_f40,9,40,2,video100,24,pos,total,VC,..\data\videos\9.avi,..\data\frames\v9_f40.png,..\data\rotulos\anotacoes-tecgraf\batch-2\VC\v...
971,v9_f50,9,50,3,video100,24,pos,total,BB,..\data\videos\9.avi,..\data\frames\v9_f50.png,..\data\rotulos\anotacoes-tecgraf\batch-3\BB\v...


In [10]:
# Salvando dados de interesse
save_video_frame_metadata_to_csv(video_frame_df, filename='video_frame_metadata.csv', output_dir='../data/metadados/')

Video frame Metadata saved to ../data/metadados/video_frame_metadata.csv


### **Salvando Frames de Interesse para uso futuro**

In [11]:
# Salva frames de que foram reotulados

def check_empty_folder(folder_path):
    if os.listdir(folder_path):  # Se a pasta NÃO estiver vazia
        raise RuntimeError(f"A pasta '{folder_path}' não está vazia! Esvazie caso queira um reprocessamento.")
    else:
        print(f"A pasta '{folder_path}' está vazia.")


check_empty_folder(frames_dir)

for i in tqdm(range(len(video_frame_df)), desc = "Frames Salvos:"):
    df_aux = video_frame_df.loc[i]
    frame_id = int(df_aux["frame_id"])
    video_path = df_aux["video_path"]
    frame_path = df_aux["frame_path"]

    frame = get_frame_from_video(video_path, frame_id)
    plt.imsave(frame_path, frame)


A pasta '..\data\frames\' está vazia.


Frames Salvos:: 100%|██████████| 973/973 [01:42<00:00,  9.47it/s]


### **Breve analise dos dados rotulados filtrados**

In [26]:
# Quantidade de frames rotulados por vídeo

fig = px.histogram(
    video_frame_df.video_id,
    title='Quantidade de frames rotulados por vídeo',
    histnorm='probability density',
    text_auto=True
)

fig.update_layout(
    bargap=0.2,
    yaxis_tickformat=',.0%',
    width=800,
    height=500
)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'histnorm': 'probability density',
              'hovertemplate': 'variable=video_id<br>value=%{x}<br>probability density=%{y}<extra></extra>',
              'legendgroup': 'video_id',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'video_id',
              'orientation': 'v',
              'showlegend': True,
              'texttemplate': '%{value}',
              'type': 'histogram',
              'x': array(['100', '100', '100', ..., '9', '9', '9'], shape=(973,), dtype=object),
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'bargap': 0.2,
               'barmode': 'relative',
               'height': 500,
               'legend': {'title': {'text': 'variable'}, 'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Quantidade de frames rotulados por vídeo'},
               'width': 800,
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'value'}},
               'yaxis': {'anchor': 'x',
                         'domain': [0.0, 1.0],
                         'tickformat': ',.0%',
                         'title': {'text': 'probability density'}}}
})

In [22]:
# Número de vídeos com a mesma quantidade de frames rotulados

fig = px.histogram(
    video_frame_df.video_id.value_counts(), 
    title='Número de vídeos com a mesma quantidade de frames rotulados',
    histnorm='probability density',
    text_auto=True
)


fig.update_layout(
    bargap=0.2,
    yaxis_tickformat=',.0%',
    width=800,
    height=500
)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'histnorm': 'probability density',
              'hovertemplate': 'variable=count<br>value=%{x}<br>probability density=%{y}<extra></extra>',
              'legendgroup': 'count',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'count',
              'orientation': 'v',
              'showlegend': True,
              'texttemplate': '%{value}',
              'type': 'histogram',
              'x': {'bdata': ('BQUFBQUFBQUFBQUFBQUFBQUFBQUFBQ' ... 'QEBAQEBAQDAwMDAwMDAwMDAwMDAwIC'),
                    'dtype': 'i1'},
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'bargap': 0.2,
               'barmode': 'relative',
               'height': 500,
               'legend': {'title': {'text': 'variable'}, 'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Número de vídeos com a mesma quantidade de frames rotulados'},
               'width': 800,
               'xaxis': {'anchor': 'y', 'domain': [0.0, 1.0], 'title': {'text': 'value'}},
               'yaxis': {'anchor': 'x',
                         'domain': [0.0, 1.0],
                         'tickformat': ',.0%',
                         'title': {'text': 'probability density'}}}
})

In [23]:
# Quantidade de frames rotulados por rotuladores

fig = px.histogram(
    video_frame_df.selected_labeler, 
    title='Quantidade de frames rotulados por labeler',
    histnorm='probability density',
    text_auto=True
)

fig.update_xaxes(categoryorder='total descending')

fig.update_layout(
    bargap=0.2,
    yaxis_tickformat=',.0%',
    width=800,
    height=500, 
)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'histnorm': 'probability density',
              'hovertemplate': ('variable=selected_labeler<br>v' ... 'ty density=%{y}<extra></extra>'),
              'legendgroup': 'selected_labeler',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': 'selected_labeler',
              'orientation': 'v',
              'showlegend': True,
              'texttemplate': '%{value}',
              'type': 'histogram',
              'x': array([np.str_('CS'), np.str_('BB'), np.str_('AM'), ..., np.str_('CS'),
                          np.str_('BB'), np.str_('AM')], shape=(973,), dtype=object),
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'bargap': 0.2,
               'barmode': 'relative',
               'height': 500,
               'legend': {'title': {'text': 'variable'}, 'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Quantidade de frames rotulados por labeler'},
               'width': 800,
               'xaxis': {'anchor': 'y',
                         'categoryorder': 'total descending',
                         'domain': [0.0, 1.0],
                         'title': {'text': 'value'}},
               'yaxis': {'anchor': 'x',
                         'domain': [0.0, 1.0],
                         'tickformat': ',.0%',
                         'title': {'text': 'probability density'}}}
})

In [24]:
# Frequência dos momentos (pre/pos) 

fig = px.histogram(
    patients_df,
    x='momento',
    title='Distribuição de vídeos por momento',
    text_auto=True,
    category_orders={'momento': ['pre', 'pos']}
)

fig.update_layout(
    bargap=0.2,
    width=600,
    height=500
)

ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

Figure({
    'data': [{'bingroup': 'x',
              'hovertemplate': 'momento=%{x}<br>count=%{y}<extra></extra>',
              'legendgroup': '',
              'marker': {'color': '#636efa', 'pattern': {'shape': ''}},
              'name': '',
              'orientation': 'v',
              'showlegend': False,
              'texttemplate': '%{value}',
              'type': 'histogram',
              'x': array(['pos', 'pos', 'pos', ..., 'pos', 'pos', 'pos'],
                         shape=(236,), dtype=object),
              'xaxis': 'x',
              'yaxis': 'y'}],
    'layout': {'bargap': 0.2,
               'barmode': 'relative',
               'height': 500,
               'legend': {'tracegroupgap': 0},
               'template': '...',
               'title': {'text': 'Distribuição de vídeos por momento'},
               'width': 600,
               'xaxis': {'anchor': 'y',
                         'categoryarray': [pre, pos],
                         'categoryorder': 'array',
                         'domain': [0.0, 1.0],
                         'title': {'text': 'momento'}},
               'yaxis': {'anchor': 'x', 'domain': [0.0, 1.0], 'title': {'text': 'count'}}}
})

In [25]:
# Distribuição de vídeos por tipo de exame

fig = px.histogram(
    patients_df[patients_df.momento == 'pos'],
    x='procedimento',
    title='Distribuição de vídeos por tipo de exame',
    text_auto=True,
)

fig.update_layout(
    bargap=0.2,
    width=800,
    height=500
)
fig.show()

print(patients_df[patients_df.momento == 'pos'].procedimento.value_counts(normalize=True))


ValueError: Mime type rendering requires nbformat>=4.2.0 but it is not installed

In [ ]:
# Plot a distribuição dos pacientes
distribuicao_pacientes = (
    video_frame_df
    .paciente_id
    .value_counts()
    .to_frame()
    .reset_index()
)
distribuicao_pacientes.columns = ['paciente_id', 'qtt_frames']
# Change paciente_id to category
distribuicao_pacientes['paciente_id'] = distribuicao_pacientes['paciente_id'].astype('str')
#distribuicao_pacientes = distribuicao_pacientes.head(20)

fig = px.histogram(
    distribuicao_pacientes,
    y='paciente_id',
    x='qtt_frames',
    title='Distribuição da quantidade de frames por paciente',
    # histnorm='probability density',
    text_auto=True,
    orientation='h'
)

# Sort bars in descending order
fig.update_yaxes(categoryorder='total ascending')

fig.update_layout(
    bargap=0.2,
    width=800,
    height=500
)

fig.show()